In [1]:
# In the reference dataset preparation I follow Kosugi paper (link). Further text is taken from the publication:
# 1. Remove DELs and INSs from DGV dataset with low (< 1kb) or undefined length
# 2. removed DELs, DUPs, and INVs with ≧ 95% reciprocal overlap 
#    (≧ 90% reciprocal overlap for > 1 kb variants) in the DGV and long read/assembly data, resulting in the removal of 450 variants in total.
# 3. The merge of both the datasets was conducted by removing shorter ones of overlapped DELs 
#    with ≧ 70% reciprocal overlap, resulting in the inclusion of 1671 DELs, 979 INSs, 2611 DUPs, and 233 INVs specific to the DGV SV data.

In [2]:
import os

workdir = '/vast/ai_projects/Papenfuss_SVEnsemble2/reference/'
os.chdir(workdir)
os.getcwd()

'/vast/ai_projects/Papenfuss_SVEnsemble2/benchmark'

In [3]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import pyranges as pr

In [4]:
pd.set_option('display.max_columns', None)

In [144]:
dgv_path = 'data/NA12878.DGV.hg38.tsv'
long_path = 'data/NA12878.long.hg38.tsv'

In [145]:
dgv_path = '../downloads/GRCh38_hg38_variants_2020-02-25.txt'

In [146]:
# dgv_path = '../downloads/GRCh38_hg38_variants_2016-05-15.txt'

In [147]:
dgv = pd.read_csv(dgv_path, sep='\t', dtype={'chr': 'string'})

In [148]:
dgv = dgv.dropna(subset=['chr', 'samples'])
dgv = dgv[dgv.samples.str.contains('NA12878')]

In [149]:
dgv.loc[:, 'chr'] = 'chr' + dgv['chr']

In [150]:
dgv = dgv.rename(columns={'chr': 'Chromosome', 'start': 'Start', 'end': 'End'})

In [151]:
dgv.loc[:, 'variantsubtype'] = dgv.variantsubtype.str.replace('loss', 'deletion')

In [152]:
dgv['variantsubtype'].value_counts()

variantsubtype
insertion                   12938
deletion                    10505
duplication                  2506
novel sequence insertion     1528
alu insertion                1265
alu deletion                 1257
mobile element insertion      927
gain                          681
gain+deletion                 489
line1 deletion                362
inversion                     296
sva insertion                 202
tandem duplication            182
mobile element deletion       168
sva deletion                  161
herv deletion                  38
herv insertion                 23
complex                        13
Name: count, dtype: int64

In [153]:
# sum((dgv.length <= 1000) & (dgv.variantsubtype.str.match('insertion|deletion')))

In [154]:
# 1. Remove DELs and INSs from DGV dataset with low (< 1kb) or undefined length
dgv.loc[:, 'length'] = dgv.End - dgv.Start
short_insdel_idx = (dgv.length <= 1000) & (dgv.variantsubtype.str.match('insertion|deletion'))
dgv = dgv[~short_insdel_idx]

In [155]:
sum(short_insdel_idx)

19751

In [156]:
dgv

,variantaccession,Chromosome,Start,End,varianttype,variantsubtype,reference,pubmedid,method,platform,mergedvariants,supportingvariants,mergedorsample,frequency,samplesize,observedgains,observedlosses,cohortdescription,genes,samples,length
10,dgv1e59,chr1,11189,36787,CNV,duplication,1000_Genomes_Consortium_Pilot_Project,20981092,"Digital array,Oligo aCGH,PCR,Sequencing",NaN,NaN,"esv3343481,esv3364878",M,NaN,185,2.0,0.0,NaN,NaN,"NA12878,NA19239",25598
12,nsv3319178,chr1,12601,27100,CNV,duplication,Audano_et_al_2019,30661756,Sequencing,NaN,NaN,"nssv14474028,nssv14484130,nssv14473887,nssv144...",M,NaN,14,11.0,0.0,NaN,NaN,"CHM1,CHM13,HG00268,HG00514,HG00733,HG02059,HG0...",14499
24,esv3430433,chr1,38889,40687,CNV,duplication,1000_Genomes_Consortium_Pilot_Project,20981092,"Digital array,Oligo aCGH,PCR,Sequencing",NaN,NaN,essv8692283,M,NaN,185,1.0,0.0,NaN,NaN,NA12878,1798
25,nsv3325054,chr1,40901,51900,CNV,duplication,Audano_et_al_2019,30661756,Sequencing,NaN,NaN,nssv14483491,M,NaN,14,1.0,0.0,NaN,NaN,NA12878,10999
54,nsv514929,chr1,68497,70627,CNV,gain,Campbell_et_al_2011,21397061,Oligo aCGH,NaN,NaN,"nssv2866525,nssv2862715,nssv2864991,nssv285870...",M,NaN,2366,488.0,0.0,NaN,NaN,"NA06984,NA06985,NA06986,NA06989,NA06991,NA0699...",2130
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
825065,nsv442824,chrY,23821643,24084364,CNV,gain+deletion,McCarroll_et_al_2008,18776908,SNP array,NaN,NaN,"nssv1656092,nssv1656091,nssv1656023,nssv165610...",M,NaN,270,7.0,128.0,NaN,NaN,"NA06985,NA06991,NA07000,NA07019,NA07055,NA0705...",262721
825083,nsv442825,chrY,24164942,24856589,CNV,gain+deletion,McCarroll_et_al_2008,18776908,SNP array,NaN,NaN,"nssv1656286,nssv1656194,nssv1656157,nssv165617...",M,NaN,270,11.0,128.0,NaN,NaN,"NA06985,NA06991,NA07000,NA07019,NA07055,NA0705...",691647
825143,nsv442826,chrY,24894982,26029114,CNV,gain+deletion,McCarroll_et_al_2008,18776908,SNP array,NaN,NaN,"nssv1656330,nssv1656363,nssv1656361,nssv165637...",M,NaN,270,10.0,128.0,NaN,NaN,"NA06985,NA06991,NA07000,NA07019,NA07055,NA0705...",1134132
825209,nsv442827,chrY,26131545,26292396,CNV,gain+deletion,McCarroll_et_al_2008,18776908,SNP array,NaN,NaN,"nssv1656532,nssv1656440,nssv1656524,nssv165645...",M,NaN,270,8.0,128.0,NaN,NaN,"NA06985,NA06991,NA07000,NA07019,NA07055,NA0705...",160851


In [157]:
dgv['variantsubtype'].value_counts()

variantsubtype
deletion                    2712
duplication                 2506
novel sequence insertion    1528
alu insertion               1265
alu deletion                1257
insertion                    980
mobile element insertion     927
gain                         681
gain+deletion                489
line1 deletion               362
inversion                    296
sva insertion                202
tandem duplication           182
mobile element deletion      168
sva deletion                 161
herv deletion                 38
herv insertion                23
complex                       13
Name: count, dtype: int64

In [158]:
# 2. removed DELs, DUPs, and INVs with ≧ 95% reciprocal overlap 
#    (≧ 90% reciprocal overlap for > 1 kb variants) in the DGV and long read/assembly data, resulting in the removal of 450 variants in total.

dgv = pr.PyRanges(dgv)
dgv = dgv.join(dgv, report_overlap=True).df
dgv.loc[:, 'overlap_frac'] = dgv.Overlap * 2 / (dgv.length + dgv.length_b)
dgv.loc[:, 'overlapped'] = (((dgv.length + dgv.length_b) < 2000) & (dgv.overlap_frac > 0.95)) \
                               | (((dgv.length + dgv.length_b) > 2000) & (dgv.overlap_frac > 0.90))

dgv_for_rm_df = dgv[
    (dgv.variantaccession != dgv.variantaccession_b) \
    & dgv.overlapped \
    & dgv.variantaccession.isin(['deletion', 'duplication', 'inversion']) \
    & dgv.variantaccession == dgv.variantaccession_b
]
dgv_for_rm = (pd.concat([dgv_for_rm_df['variantaccession'], dgv_for_rm_df['variantaccession_b']])).unique()

dgv = dgv[~dgv['variantaccession'].isin(dgv_for_rm)]
dgv = dgv[['variantaccession', 'Chromosome', 'Start', 'End', 'variantsubtype', 'reference', 'method', 'samples', 'length']].drop_duplicates()

In [159]:
dgv['variantsubtype'].value_counts()

variantsubtype
deletion                    2712
duplication                 2506
alu deletion                1257
insertion                    980
mobile element insertion     824
gain                         681
gain+deletion                489
line1 deletion               362
inversion                    296
tandem duplication           182
mobile element deletion      168
sva deletion                 161
alu insertion                108
novel sequence insertion      92
herv deletion                 38
sva insertion                 17
complex                       13
herv insertion                 5
Name: count, dtype: int64

In [160]:
long = pd.read_csv(long_path, sep='\t')
long = long.rename(columns={'Chrom': 'Chromosome'})
long.loc[:, 'length'] = long.End - long.Start

In [161]:
long['Event Type'].value_counts()

Event Type
insertion    13017
deletion      7784
Name: count, dtype: int64

In [162]:
long = long.reset_index()
long = pr.PyRanges(long)
long = long.join(long, report_overlap=True).df
long.loc[:, 'overlap_frac'] = long.Overlap * 2 / (long.length + long.length_b)
long.loc[:, 'overlapped'] = (((long.length + long.length_b) < 2000) & (long.overlap_frac > 0.95)) \
                               | (((long.length + long.length_b) > 2000) & (long.overlap_frac > 0.90))
long_for_rm_df = long[
    (long.index != long.index_b) \
    & long.overlapped \
    & long['Event Type'].isin(['deletion', 'duplication', 'inversion']) \
    & long['Event Type'] == long['Event Type_b']
]
long_for_rm = (pd.concat([long_for_rm_df['index'], long_for_rm_df['index_b']])).unique()

long = long[~long['index'].isin(long_for_rm)]
long = long[['index', 'Chromosome', 'Start', 'End', 'Event Type', 'length']].drop_duplicates()

In [163]:
dgv_pr = pr.PyRanges(dgv)
long_pr = pr.PyRanges(long)
joined_dgv_long = long_pr.join(dgv_pr, report_overlap=True)
joined_dgv_long = joined_dgv_long.df
joined_dgv_long.loc[:, 'overlap_frac'] = joined_dgv_long.Overlap * 2 / (joined_dgv_long.End - joined_dgv_long.Start + joined_dgv_long.End_b - joined_dgv_long.Start_b)

# IDs for vars with any overlaps will be used to exclude them from the other datasets before merging
# index_overlapping = joined_dgv_long['index'].unique()
# variantaccession_overlapping = joined_dgv_long['variantaccession'].unique()

dgv_shorter = joined_dgv_long.length_b < joined_dgv_long.length
long_rm_idx = ~dgv_shorter & (joined_dgv_long.overlap_frac > 0.70)
dgv_rm_idx = dgv_shorter & (joined_dgv_long.overlap_frac > 0.70)
long_rm = joined_dgv_long[~long_rm_idx]['index'].unique()
dgv_rm = joined_dgv_long[~dgv_rm_idx]['variantaccession'].unique()
# joined_dgv_long = joined_dgv_long[~long_rm & ~dgv_rm]

# dgv_nonoverlap = dgv[~dgv.variantaccession.isin(variantaccession_overlapping)]

# long_nonoverlap = long[~long['index'].isin(variantaccession_overlapping)]

# joined_dgv_long = pd.concat([joined_dgv_long, dgv_nonoverlap, long_nonoverlap])

In [164]:
dgv_rename_dict = {
    'variantsubtype': 'ALT'
}
long_rename_dict = {
    'Event Type': 'ALT'
}

In [165]:
dgv = dgv.rename(columns=dgv_rename_dict)
long = long.rename(columns=long_rename_dict)

In [166]:
dgv_to_join = dgv[~dgv.variantaccession.isin(dgv_rm)]
dgv_to_join.loc[:, 'source'] = 'dgv'
dgv_to_join.loc[:, 'ID'] = dgv_to_join['variantaccession']
dgv_to_join = dgv_to_join.drop(columns=['variantaccession'])

/tmp/ipykernel_2721265/45942989.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dgv_to_join.loc[:, 'source'] = 'dgv'
/tmp/ipykernel_2721265/45942989.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dgv_to_join.loc[:, 'ID'] = dgv_to_join['variantaccession']


In [167]:
long_to_join = long[~long['index'].isin(long_rm)]
long_to_join.loc[:, 'source'] = 'long'
long_to_join.loc[:, 'ID'] = 'long' + long_to_join['index'].astype(str)
long_to_join = long_to_join.drop(columns=['index'])

/tmp/ipykernel_2721265/1659464836.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  long_to_join.loc[:, 'source'] = 'long'
/tmp/ipykernel_2721265/1659464836.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  long_to_join.loc[:, 'ID'] = 'long' + long_to_join['index'].astype(str)


In [168]:
full = pd.concat([dgv_to_join, long_to_join])
full

,Chromosome,Start,End,ALT,reference,method,samples,length,source,ID
0,chr1,11189,36787,duplication,1000_Genomes_Consortium_Pilot_Project,"Digital array,Oligo aCGH,PCR,Sequencing","NA12878,NA19239",25598,dgv,dgv1e59
2,chr1,12601,27100,duplication,Audano_et_al_2019,Sequencing,"CHM1,CHM13,HG00268,HG00514,HG00733,HG02059,HG0...",14499,dgv,nsv3319178
4,chr1,38889,40687,duplication,1000_Genomes_Consortium_Pilot_Project,"Digital array,Oligo aCGH,PCR,Sequencing",NA12878,1798,dgv,esv3430433
5,chr1,40901,51900,duplication,Audano_et_al_2019,Sequencing,NA12878,10999,dgv,nsv3325054
6,chr1,68497,70627,gain,Campbell_et_al_2011,Oligo aCGH,"NA06984,NA06985,NA06986,NA06989,NA06991,NA0699...",2130,dgv,nsv514929
...,...,...,...,...,...,...,...,...,...,...
23715,chrX,156021581,156022832,deletion,NaN,NaN,NaN,1251,long,long20796
23716,chrX,156029094,156029194,insertion,NaN,NaN,NaN,100,long,long20797
23717,chrX,156029274,156029375,insertion,NaN,NaN,NaN,101,long,long20798
23718,chrX,156030012,156030115,insertion,NaN,NaN,NaN,103,long,long20799


In [169]:
# Finally, GIAB svclassify DELs and INSs are added

In [170]:
giab_dels_path = 'data/Personalis_1000_Genomes_deduplicated_deletions.bed'
giab_ins_path = 'data/Spiral_Genetics_insertions.bed'

In [171]:
giab_dels = pd.read_csv(giab_dels_path, sep='\t')
giab_ins = pd.read_csv(giab_ins_path, sep='\t')

In [172]:
giab_dels.loc[:, 'ALT'] = 'deletion'
giab_ins.loc[:, 'ALT'] = 'insertion'

In [173]:
giab = pd.concat([giab_dels, giab_ins]).reset_index(drop=True)
giab = giab.rename(columns={'Chr': 'Chromosome'})
giab.loc[:, 'Chromosome'] = 'chr' + giab.Chromosome.astype(str)
giab.loc[:, 'source'] = 'giab'
giab.loc[:, 'ID'] = 'giab' + giab.index.astype(str)
giab.loc[:, 'length'] = giab.End - giab.Start
giab

,Chromosome,Start,End,ALT,source,ID,length
0,chr1,1598414,1598580,deletion,giab,giab0,166
1,chr1,2041460,2041556,deletion,giab,giab1,96
2,chr1,2911537,2911850,deletion,giab,giab2,313
3,chr1,2919032,2919366,deletion,giab,giab3,334
4,chr1,7570072,7571526,deletion,giab,giab4,1454
...,...,...,...,...,...,...,...
2739,chr19,49086216,49086433,insertion,giab,giab2739,217
2740,chr20,7699427,7699659,insertion,giab,giab2740,232
2741,chr21,34529178,34529234,insertion,giab,giab2741,56
2742,chr22,49094501,49094673,insertion,giab,giab2742,172


In [174]:
giab_pr = pr.PyRanges(giab)
full_pr = pr.PyRanges(full)

In [175]:
full_pr

,Chromosome,Start,End,ALT,reference,method,samples,length,source,ID
0,chr1,11189,36787,duplication,1000_Genomes_Consortium_Pilot_Project,"Digital array,Oligo aCGH,PCR,Sequencing","NA12878,NA19239",25598,dgv,dgv1e59
1,chr1,12601,27100,duplication,Audano_et_al_2019,Sequencing,"CHM1,CHM13,HG00268,HG00514,HG00733,HG02059,HG0...",14499,dgv,nsv3319178
2,chr1,38889,40687,duplication,1000_Genomes_Consortium_Pilot_Project,"Digital array,Oligo aCGH,PCR,Sequencing",NA12878,1798,dgv,esv3430433
3,chr1,40901,51900,duplication,Audano_et_al_2019,Sequencing,NA12878,10999,dgv,nsv3325054
4,chr1,68497,70627,gain,Campbell_et_al_2011,Oligo aCGH,"NA06984,NA06985,NA06986,NA06989,NA06991,NA0699...",2130,dgv,nsv514929
...,...,...,...,...,...,...,...,...,...,...
21893,chrY,23821643,24084364,gain+deletion,McCarroll_et_al_2008,SNP array,"NA06985,NA06991,NA07000,NA07019,NA07055,NA0705...",262721,dgv,nsv442824
21894,chrY,24164942,24856589,gain+deletion,McCarroll_et_al_2008,SNP array,"NA06985,NA06991,NA07000,NA07019,NA07055,NA0705...",691647,dgv,nsv442825
21895,chrY,24894982,26029114,gain+deletion,McCarroll_et_al_2008,SNP array,"NA06985,NA06991,NA07000,NA07019,NA07055,NA0705...",1134132,dgv,nsv442826
21896,chrY,26131545,26292396,gain+deletion,McCarroll_et_al_2008,SNP array,"NA06985,NA06991,NA07000,NA07019,NA07055,NA0705...",160851,dgv,nsv442827


In [176]:
giab_pr_overlapping = giab_pr.join(full_pr)
giab_rm = giab_pr_overlapping.ID.unique()
giab = giab[~giab.ID.isin(giab_rm)]
giab

,Chromosome,Start,End,ALT,source,ID,length
0,chr1,1598414,1598580,deletion,giab,giab0,166
1,chr1,2041460,2041556,deletion,giab,giab1,96
2,chr1,2911537,2911850,deletion,giab,giab2,313
3,chr1,2919032,2919366,deletion,giab,giab3,334
4,chr1,7570072,7571526,deletion,giab,giab4,1454
...,...,...,...,...,...,...,...
2739,chr19,49086216,49086433,insertion,giab,giab2739,217
2740,chr20,7699427,7699659,insertion,giab,giab2740,232
2741,chr21,34529178,34529234,insertion,giab,giab2741,56
2742,chr22,49094501,49094673,insertion,giab,giab2742,172


In [177]:
full = pd.concat([full, giab])
full

,Chromosome,Start,End,ALT,reference,method,samples,length,source,ID
0,chr1,11189,36787,duplication,1000_Genomes_Consortium_Pilot_Project,"Digital array,Oligo aCGH,PCR,Sequencing","NA12878,NA19239",25598,dgv,dgv1e59
2,chr1,12601,27100,duplication,Audano_et_al_2019,Sequencing,"CHM1,CHM13,HG00268,HG00514,HG00733,HG02059,HG0...",14499,dgv,nsv3319178
4,chr1,38889,40687,duplication,1000_Genomes_Consortium_Pilot_Project,"Digital array,Oligo aCGH,PCR,Sequencing",NA12878,1798,dgv,esv3430433
5,chr1,40901,51900,duplication,Audano_et_al_2019,Sequencing,NA12878,10999,dgv,nsv3325054
6,chr1,68497,70627,gain,Campbell_et_al_2011,Oligo aCGH,"NA06984,NA06985,NA06986,NA06989,NA06991,NA0699...",2130,dgv,nsv514929
...,...,...,...,...,...,...,...,...,...,...
2739,chr19,49086216,49086433,insertion,NaN,NaN,NaN,217,giab,giab2739
2740,chr20,7699427,7699659,insertion,NaN,NaN,NaN,232,giab,giab2740
2741,chr21,34529178,34529234,insertion,NaN,NaN,NaN,56,giab,giab2741
2742,chr22,49094501,49094673,insertion,NaN,NaN,NaN,172,giab,giab2742


In [178]:
na12878_path = 'saved/NA12878.reference.tsv'
full.to_csv(na12878_path, sep='\t')